# Fraud Detection Model

In [17]:
# Data Handeling And cleaning
import pandas as pd
import numpy as np
import datetime

# Visulization
import matplotlib.pyplot as plt
import plotly as pl

# Machine Learning
import sklearn

# Spliting Dataset
from sklearn.model_selection import train_test_split

# Encoding
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder

# Feature Scaling
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, AdaBoostClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# Model Evaluation
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score

In [ ]:
pip install xgboost

## Step 2: Cleaning Data

In [18]:
data = pd.read_csv('customer_analytics_dataset.csv')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   customer_id         5000 non-null   object 
 1   age                 5000 non-null   int64  
 2   gender              5000 non-null   object 
 3   country             5000 non-null   object 
 4   avg_order_value     4750 non-null   float64
 5   total_orders        5000 non-null   int64  
 6   last_purchase       5000 non-null   int64  
 7   is_fraudulent       5000 non-null   int64  
 8   preferred_category  5000 non-null   object 
 9   email_open_rate     4750 non-null   float64
 10  customer_since      5000 non-null   object 
 11  loyalty_score       5000 non-null   int64  
 12  churn_risk          5000 non-null   float64
dtypes: float64(3), int64(5), object(5)
memory usage: 507.9+ KB


In [19]:
data.isna().sum()

customer_id             0
age                     0
gender                  0
country                 0
avg_order_value       250
total_orders            0
last_purchase           0
is_fraudulent           0
preferred_category      0
email_open_rate       250
customer_since          0
loyalty_score           0
churn_risk              0
dtype: int64

In [20]:
data.shape

(5000, 13)

In [21]:
data.select_dtypes('number').shape

(5000, 8)

In [22]:
for col in data.select_dtypes('number').columns:
    if data[col].isna().sum() > 0:
        data[col].fillna(data[col].median(), inplace=True)   #we remove fill all nan with meadian bcz there is outlier thath why we fill median

In [23]:
data.isna().sum()

customer_id           0
age                   0
gender                0
country               0
avg_order_value       0
total_orders          0
last_purchase         0
is_fraudulent         0
preferred_category    0
email_open_rate       0
customer_since        0
loyalty_score         0
churn_risk            0
dtype: int64

In [24]:
data.drop('customer_id', axis=1, inplace = True)

In [25]:
data.sample(5)

,age,gender,country,avg_order_value,total_orders,last_purchase,is_fraudulent,preferred_category,email_open_rate,customer_since,loyalty_score,churn_risk
1754,58,Female,UK,44.56,11,250,0,Electronics,78.8,2024-02-05,15,0.07
3433,30,Female,Germany,123.86,6,260,0,Electronics,9.8,2021-04-11,20,0.69
1093,28,Male,China,206.84,8,324,0,Home,49.3,2024-01-26,75,0.54
3078,38,Female,China,217.61,13,204,0,Beauty,73.7,2024-08-25,22,0.56
2932,49,Female,Brazil,31.94,11,99,0,Beauty,35.0,2024-11-20,26,0.21


In [26]:
data.select_dtypes(['datetime','number']).corr()['is_fraudulent']

age                0.003242
avg_order_value   -0.010311
total_orders       0.002200
last_purchase     -0.007725
is_fraudulent      1.000000
email_open_rate    0.016458
loyalty_score      0.003803
churn_risk        -0.003543
Name: is_fraudulent, dtype: float64

In [27]:
data.customer_since.dtype

dtype('O')

In [28]:
data['customer_since'] = pd.to_datetime(data['customer_since'], errors='coerce' )
data.customer_since.dtype

dtype('<M8[ns]')

In [29]:
dt = datetime.datetime

In [30]:
data['customer_since'].dt.year.value_counts()

customer_since
2024    1042
2023    1006
2022     964
2021     959
2020     542
2025     487
Name: count, dtype: int64

In [31]:
data['customer_since'].dt.to_period('M').nunique()

61

In [32]:
data['customer_since_year'] = data['customer_since'].dt.year

In [33]:
del data['customer_since']

In [34]:
data.sample(2)

,age,gender,country,avg_order_value,total_orders,last_purchase,is_fraudulent,preferred_category,email_open_rate,loyalty_score,churn_risk,customer_since_year
740,71,Male,China,40.94,11,186,0,Home,84.5,45,0.50,2022
3233,24,Male,India,73.72,8,352,0,Fashion,97.2,20,0.24,2022


## Step 2: Train Test Split

In [35]:
X = data.drop('is_fraudulent', axis =1)
y = data.is_fraudulent

In [36]:
x_train, x_test, y_train, y_test = train_test_split(X,y , test_size=0.2, random_state=42)
x_train.shape, y_train.shape, x_test.shape, y_test.shape

((4000, 11), (4000,), (1000, 11), (1000,))

We Create A fucntion except using pycaret to checking my cooding knowledge so i create function in whoch i pass my x and y 
variable and then the futhure work and prediction automaticaally model do

In [86]:
def train_ml_model_by_applying_preprocessing_and_cross_validate(x,y):
    
    # Train Test Split
    x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
    
    # Encoding
    categorical_columns=['gender','country','preferred_category']
    ohe=OneHotEncoder(sparse_output=False,handle_unknown='ignore')
    x_train_encoded=ohe.fit_transform(x_train[categorical_columns])
    x_test_encoded=ohe.transform(x_test[categorical_columns])

    # Convert encoded arrays to DataFrame
    x_train_encoded=pd.DataFrame(x_train_encoded,columns=ohe.get_feature_names_out(categorical_columns),index=x_train.index)
    x_test_encoded=pd.DataFrame(x_test_encoded,columns=ohe.get_feature_names_out(categorical_columns),index=x_test.index)
    
    # Remove original categorical columns
    x_train=x_train.drop(columns=categorical_columns)
    x_test=x_test.drop(columns=categorical_columns)
    
    # Combine numerical + encoded categorical
    x_train=pd.concat([x_train,x_train_encoded],axis=1)
    x_test=pd.concat([x_test,x_test_encoded],axis=1)

    # SMOTE
    from imblearn.over_sampling import SMOTE
    ss = SMOTE()
    x_train, y_train = ss.fit_resample(x_train,y_train)

    
    # Scaling
    ss=StandardScaler()
    ss.set_output(transform='pandas')
    x_train_scalar=ss.fit_transform(x_train)
    x_test_scalar=ss.transform(x_test)
    mn=MinMaxScaler()
    mn.set_output(transform='pandas')
    x_train_mm=mn.fit_transform(x_train)
    x_test_mm=mn.transform(x_test)
    
    # Models
    dic_of_models={
        'Logistic_Regression':LogisticRegression(),
        'Support Vector Classifier(SVC)':SVC(),
        'DecisionTreeClassifier':DecisionTreeClassifier(),
        'RandomForestClassifier':RandomForestClassifier(),
        'BaggingClassifier':BaggingClassifier(),
        'AdaBoostClassifier':AdaBoostClassifier(),
        'GradientBoostingClassifier':GradientBoostingClassifier()
    }
    
    # Scaled Data
    scaled_data={
        "MinMaxScaler":(x_train_mm,x_test_mm),
        "StandardScaler":(x_train_scalar,x_test_scalar)
    }

    # Training
    results=[]
    for scaler_name,(X_train,X_test) in scaled_data.items():
        for model_name,model in dic_of_models.items():
            model.fit(X_train,y_train)
            train_pred=model.predict(X_train)
            test_pred=model.predict(X_test)
            results.append({
                'Scaler':scaler_name,
                'Model_Name':model_name,
                'Accuracy_Train':accuracy_score(y_train,train_pred),
                'Accuracy_Test':accuracy_score(y_test,test_pred),
                'Precision_Train':precision_score(y_train,train_pred),
                'Precision_Test':precision_score(y_test,test_pred),
                'Recall_Train':recall_score(y_train,train_pred),
                'Recall_Test':recall_score(y_test,test_pred)
            })
    results_df=pd.DataFrame(results)
    return results_df

In [87]:
results_df=train_ml_model_by_applying_preprocessing_and_compare_models(X,y)
results_df

,Scaler,Model_Name,Accuracy_Train,Accuracy_Test,Precision_Train,Precision_Test,Recall_Train,Recall_Test
0,MinMaxScaler,Logistic_Regression,0.626123,0.590,0.619209,0.040670,0.655119,0.653846
1,MinMaxScaler,Support Vector Classifier(SVC),0.971901,0.957,0.989356,0.052632,0.954067,0.038462
2,MinMaxScaler,DecisionTreeClassifier,1.000000,0.939,1.000000,0.051282,1.000000,0.076923
3,MinMaxScaler,RandomForestClassifier,1.000000,0.974,1.000000,0.000000,1.000000,0.000000
4,MinMaxScaler,BaggingClassifier,0.997049,0.968,1.000000,0.000000,0.994098,0.000000
5,MinMaxScaler,AdaBoostClassifier,0.983064,0.974,1.000000,0.000000,0.966128,0.000000
6,MinMaxScaler,GradientBoostingClassifier,0.983834,0.974,1.000000,0.000000,0.967667,0.000000
7,MinMaxScaler,XGBClassifier,1.000000,0.973,1.000000,0.000000,1.000000,0.000000
8,StandardScaler,Logistic_Regression,0.626123,0.591,0.619094,0.040767,0.655633,0.653846
9,StandardScaler,Support Vector Classifier(SVC),0.992173,0.945,0.988538,0.000000,0.995894,0.000000


In [88]:
best_model = results_df.loc[
    results_df["Recall_Test"].idxmax()
]

print(best_model)  #this is our best fit model

Scaler                    MinMaxScaler
Model_Name         Logistic_Regression
Accuracy_Train                0.626123
Accuracy_Test                     0.59
Precision_Train               0.619209
Precision_Test                 0.04067
Recall_Train                  0.655119
Recall_Test                   0.653846
Name: 0, dtype: object


In this project, multiple machine learning classification models were trained for fraud detection after applying data preprocessing, categorical encoding, feature scaling, and SMOTE to handle class imbalance. SMOTE was applied only to the training data, while the test data was kept unchanged for unbiased evaluation. Different models were compared using Accuracy, Precision, and Recall. Since fraud detection requires identifying as many fraudulent transactions as possible, Recall was given higher importance when selecting the best-performing model. Based on the experimental results, the model with the best test Recall and an acceptable Precision was selected as the final model.